# Correlation and Exploratory Analysis

We have just read about correlation as a concept: what it measures, what it misses, and why two variables moving together does not mean one causes the other. Now we are going to work with it directly.

The dataset here contains around 3,000 property sales. Our job in this notebook is the same job any analyst faces at the start of a project: look at the data, find out which features relate to price, and start separating genuine signal from misleading patterns. By the end, we will have a clear picture of which variables are worth taking forward into a regression model.

We will:

- Create **scatter plots** to visualise relationships between pairs of variables
- Calculate and interpret a **correlation matrix** using `df.corr()`
- Display correlation as a **heatmap** with seaborn
- Explain the difference between **Pearson** and **Spearman** correlation
- Recognise why **correlation does not equal causation**

In [ ]:
%pip install -q pandas matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [ ]:
df = pd.read_csv("../data/property_sales.csv")
nbr = pd.read_csv("../data/neighbourhood_features.csv")

print(df.shape)
df.head()

The dataset contains around 3,000 property sales with numeric and categorical features. `sale_price` is the variable we ultimately want to predict. Before building any model, though, we need to understand how the other columns relate to it.

---

## 1. Scatter plots: visualising pairwise relationships

The correlation coefficient gives us a number, but a number on its own can mislead. A scatter plot lets us see the shape of a relationship before we try to summarise it. That matters because correlation only captures *linear* patterns, and we need to know whether a straight line is the right summary in the first place.

### Price vs floor area

Start with the most intuitive relationship. Larger properties should cost more. The question is how strong and how consistent that pattern is.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["floor_area_sqm"], df["sale_price"], alpha=0.3, s=10)
plt.xlabel("Floor area (sqm)")
plt.ylabel("Sale price (£)")
plt.title("Sale price vs floor area")
plt.tight_layout()
plt.show()

There is a clear upward trend: bigger properties sell for more. But notice the spread. Two properties with the same floor area can differ by hundreds of thousands of pounds. Floor area is part of the story, not the whole story.

### Price vs distance to station

Now consider a feature where the expected direction is reversed. Properties closer to a train station are often more desirable, so we would expect price to *decrease* as distance increases.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["distance_to_station_km"], df["sale_price"], alpha=0.3, s=10)
plt.xlabel("Distance to station (km)")
plt.ylabel("Sale price (£)")
plt.title("Sale price vs distance to station")
plt.tight_layout()
plt.show()

The negative trend is there, but it is weaker and noisier than the floor area relationship. If we were asked to draw a line through this cloud by hand, we would not feel very confident about its slope. That uncertainty is something the correlation coefficient will quantify for us shortly.

### Adding colour to show a third variable

A scatter plot of two variables can hide important structure. Adding colour for a categorical variable like `neighbourhood` reveals whether the overall pattern holds everywhere or whether certain areas sit consistently higher or lower.

In [ ]:
# Pick 6 neighbourhoods for clarity
sample_nbrs = df["neighbourhood"].value_counts().index[:6]
subset = df[df["neighbourhood"].isin(sample_nbrs)]

plt.figure(figsize=(10, 6))
for nbr_name in sample_nbrs:
    mask = subset["neighbourhood"] == nbr_name
    plt.scatter(
        subset.loc[mask, "floor_area_sqm"],
        subset.loc[mask, "sale_price"],
        alpha=0.4, s=15, label=nbr_name,
    )
plt.xlabel("Floor area (sqm)")
plt.ylabel("Sale price (£)")
plt.title("Sale price vs floor area by neighbourhood")
plt.legend(title="Neighbourhood", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

Some neighbourhoods sit consistently above the overall trend; others sit below. This is our first hint that neighbourhood has its own effect on price, independent of floor area. Keep this in mind. When we build regression models later, we will need to account for it.

---

## 2. The correlation matrix

Scatter plots are good for examining one relationship at a time, but this dataset has many numeric features. We need a way to scan all the pairwise relationships at once. That is what the **correlation matrix** does.

We already know from the explainer that the correlation coefficient *r* ranges from -1 to +1 and measures the strength and direction of a linear relationship. Pandas computes the full matrix in a single call.

| Value of *r* | Interpretation |
|---|---|
| +1 | Perfect positive linear relationship |
| 0 | No linear relationship |
| -1 | Perfect negative linear relationship |

In [ ]:
numeric_cols = [
    "sale_price", "bedrooms", "bathrooms", "floor_area_sqm",
    "garden_sqm", "age_years", "distance_to_station_km",
    "distance_to_school_km",
]

corr_matrix = df[numeric_cols].corr()
corr_matrix.round(2)

Look at the `sale_price` row. `floor_area_sqm` and `bedrooms` should have the strongest positive correlations with price, confirming what the scatter plots showed. `distance_to_station_km` should be negative. The numbers give us a ranking of which features have the strongest linear association with price.

### Heatmap visualisation

A table of numbers is hard to scan quickly. A heatmap uses colour intensity to make the strongest correlations jump out immediately.

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
)
plt.title("Correlation matrix: property features")
plt.tight_layout()
plt.show()

Red cells indicate positive correlation; blue cells indicate negative. The diagonal is always 1.0 (every variable correlates perfectly with itself).

Notice that `bedrooms` and `floor_area_sqm` are strongly correlated with *each other*, not just with price. That makes intuitive sense: bigger properties tend to have more bedrooms. This is called **multicollinearity**, and it will matter when we build regression models in the next notebook. For now, just note which features travel together.

---

## 3. Pearson vs Spearman correlation

Everything so far has used **Pearson** correlation, which is the default in `df.corr()`. Pearson measures the strength of the *linear* relationship between two variables. If the relationship is monotonic but curved (steadily rising, but not in a straight line), Pearson will underestimate it.

**Spearman** correlation works on the *ranks* of the data rather than the raw values. It captures any monotonic relationship, whether the underlying shape is linear or not. Let's see where the distinction matters in this dataset.

In [ ]:
pearson = df["sale_price"].corr(df["age_years"])
spearman = df["sale_price"].corr(df["age_years"], method="spearman")

print(f"Pearson correlation (price vs age):  {pearson:.3f}")
print(f"Spearman correlation (price vs age): {spearman:.3f}")

Property age has a non-linear effect on price. Newer properties generally cost more, but very old properties (over 100 years) gain a "character premium" that pulls prices back up. Spearman handles this better than Pearson because it only cares about the rank ordering, not the exact shape.

**Rule of thumb:** use Pearson when we expect a straight-line relationship; use Spearman when the relationship might curve but still goes consistently in one direction. The scatter plot below shows why this distinction matters here.

In [ ]:
# Visualise the non-linear age effect
plt.figure(figsize=(8, 5))
plt.scatter(df["age_years"], df["sale_price"], alpha=0.2, s=10)
plt.xlabel("Property age (years)")
plt.ylabel("Sale price (£)")
plt.title("Sale price vs property age")
plt.tight_layout()
plt.show()

---

## 4. Correlation does not equal causation

We covered the three mechanisms behind this in the explainer: confounders, reverse causation, and coincidence. Now let's see how confounding plays out in real data. The dataset includes neighbourhood-level features that let us investigate whether an apparent relationship between two variables is actually driven by a third.

In [ ]:
merged = df.merge(nbr, on="neighbourhood", how="left")

r_school_dist = merged["sale_price"].corr(merged["distance_to_school_km"])
r_income = merged["sale_price"].corr(merged["avg_household_income"])

print(f"Correlation: price vs distance to school = {r_school_dist:.3f}")
print(f"Correlation: price vs neighbourhood income = {r_income:.3f}")

If `distance_to_school_km` correlates with price, does proximity to a school cause higher prices? Think back to the confounding examples from the explainer. There is a plausible alternative explanation:

- Wealthier neighbourhoods have higher household incomes.
- Those neighbourhoods tend to have schools nearby (or the schools attract wealthier families).
- The apparent effect of school distance on price may actually be driven by neighbourhood income.

This is a **confounding variable** in action: income influences both price and school proximity, creating a correlation between two things that may not directly cause each other.

The three mechanisms worth remembering:

1. **Confounding** — a hidden third variable drives both. Ice cream sales and drowning rates both rise in summer because of temperature.
2. **Reverse causation** — the causal arrow points the other way. Areas with high property prices attract investment in schools, not the reverse.
3. **Coincidence** — with enough variables, some will correlate by chance.

In [ ]:
# Visualise the confounding: colour by neighbourhood income
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scatter1 = axes[0].scatter(
    merged["distance_to_school_km"], merged["sale_price"],
    c=merged["avg_household_income"], cmap="viridis", alpha=0.3, s=10,
)
axes[0].set_xlabel("Distance to school (km)")
axes[0].set_ylabel("Sale price (£)")
axes[0].set_title("Price vs school distance")
plt.colorbar(scatter1, ax=axes[0], label="Avg household income")

scatter2 = axes[1].scatter(
    merged["avg_household_income"], merged["sale_price"],
    alpha=0.3, s=10,
)
axes[1].set_xlabel("Avg household income (£)")
axes[1].set_ylabel("Sale price (£)")
axes[1].set_title("Price vs neighbourhood income")

plt.tight_layout()
plt.show()

The colour gradient in the left plot tells the story. High-income neighbourhoods (yellow points) cluster at certain distances. Once we account for income, the distance-to-school effect may weaken substantially or disappear. We will test this properly with multiple regression in the next notebook, where we can hold income constant and see what happens to the school-distance coefficient.

---

## 5. Exercises

### Exercise 1: Scatter plot of price vs garden size

Create a scatter plot with `garden_sqm` on the x-axis and `sale_price` on the y-axis. Add axis labels and a title. Describe the pattern you see in a comment.

In [ ]:
# Your code here


### Exercise 2: Spearman vs Pearson

Calculate both the Pearson and Spearman correlation between `sale_price` and `floor_area_sqm`. Print both values. Then do the same for `sale_price` and `age_years`. In a comment, explain which pair shows the biggest difference between the two methods and why.

In [ ]:
# Your code here


### Exercise 3: Correlation heatmap with neighbourhood data

Merge `df` with the `nbr` DataFrame on `neighbourhood`. Then compute and display a heatmap of the correlation matrix for these columns: `sale_price`, `avg_household_income`, `crime_rate_per_1000`, `green_space_pct`, `school_rating_avg`. Annotate the heatmap with values rounded to two decimal places.

In [ ]:
# Your code here


### Exercise 4: Identify a potential confounding variable

Using the merged dataset, find a pair of variables that correlate with each other. Then identify a third variable that could be confounding the relationship. Support your argument by calculating the relevant correlations and printing them. Write your explanation as comments in the code cell.

In [ ]:
# Your code here


---

## Summary

We now have a practical toolkit for the exploratory phase of any analysis:

- **Scatter plots** show the shape and strength of pairwise relationships
- **Colour encoding** reveals the influence of a third variable
- The **correlation matrix** and **heatmap** let us scan many relationships at once
- **Pearson** measures linear association; **Spearman** captures monotonic patterns that Pearson misses
- **Confounding variables** can create correlations between things that do not cause each other

The correlations we have found here are useful but provisional. We know that floor area, bedrooms, and station distance are associated with price. We suspect that neighbourhood income confounds some of those relationships. In the next notebook, we will move from describing relationships to modelling them, fitting regression lines that can make predictions and isolate each feature's contribution.